In [134]:
import numpy as np
import csv
import pandas as pd
import plotly
import plotly.graph_objs as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import norm

In [135]:
df = pd.read_csv('financial_data.csv', delimiter=',')

In [136]:
df

,X1,X2,X3,X4
0,-0.004552,20.00,50.0,4263.80
1,0.018496,20.00,56.0,3821.68
2,0.001036,20.00,38.0,3877.03
3,-0.002877,20.00,46.0,2378.55
4,-0.011841,50.00,70.0,2424.10
...,...,...,...,...
195,-0.011042,10.25,87.0,2557.16
196,0.001576,10.50,17.0,3813.45
197,0.000789,10.50,2.0,4228.02
198,-0.006110,10.75,92.0,2189.16


In [137]:
for col in df:
    df[col] = df[col].sort_values(ignore_index=True)

variation_series_x1 = df['X1']
variation_series_x2 = df['X2']
variation_series_x3 = df['X3']
df

,X1,X2,X3,X4
0,-0.041816,10.0,0.0,2053.64
1,-0.040567,10.0,0.0,2054.29
2,-0.033655,10.0,0.0,2063.30
3,-0.031525,10.0,0.0,2071.28
4,-0.026747,10.0,0.0,2073.68
...,...,...,...,...
195,0.026931,210.0,97.0,4248.33
196,0.029174,210.0,97.0,4259.13
197,0.031851,210.0,97.0,4261.82
198,0.039647,210.0,98.0,4263.80


Пункт 4.1

In [138]:
N = len(variation_series_x1)

In [139]:
empiric_function_x1 = np.arange(1, N+1) / N    

fig_empiric_x1 = go.Figure()
fig_empiric_x1.add_trace(go.Scatter(
    x=variation_series_x1,
    y=empiric_function_x1,
    mode='lines',
    line_shape='hv'
))

fig_empiric_x1.update_xaxes(title_text='x')
fig_empiric_x1.update_yaxes(title_text='Fn(x)')
fig_empiric_x1.update_layout(title='Эмпирическая функция распределения X1')
fig_empiric_x1.show()

In [140]:
empiric_function_x2 = np.arange(1, N+1) / N    

fig_empiric_x2 = go.Figure()
fig_empiric_x2.add_trace(go.Scatter(
    x=variation_series_x2,
    y=empiric_function_x2,
    mode='lines',
    line_shape='hv'
))

fig_empiric_x2.update_xaxes(title_text='x')
fig_empiric_x2.update_yaxes(title_text='Fn(x)')
fig_empiric_x2.update_layout(title='Эмпирическая функция распределения X2')
fig_empiric_x2.show()

In [141]:
empiric_function_x3 = np.arange(1, N+1) / N    

fig_empiric_x3 = go.Figure()
fig_empiric_x3.add_trace(go.Scatter(
    x=variation_series_x3,
    y=empiric_function_x3,
    mode='lines',
    line_shape='hv'
))

fig_empiric_x3.update_xaxes(title_text='x')
fig_empiric_x3.update_yaxes(title_text='Fn(x)')
fig_empiric_x3.update_layout(title='Эмпирическая функция распределения X3')
fig_empiric_x3.show()

In [142]:
def get_characteristics(x):
    # Среднее
    x_mean = np.mean(x)

    # Дисперсии
    S2 = np.var(x, ddof=0)
    sigma2_hat = np.var(x, ddof=1)

    #ddof - это "delta degrees of freedom". ddof=0 - делим на n (смещённая), ddof=1 - делим на n−1 (несмещённая)

    # Стандартные отклонения
    S = np.sqrt(S2)
    sigma_hat = np.sqrt(sigma2_hat)

    # Медиана
    median = np.median(x)

    # Квартили
    q25, q50, q75 = np.percentile(x, [25, 50, 75])

    print(f"x̄ = {x_mean}")
    print(f"S² = {S2}")
    print(f"σ̂² = {sigma2_hat}")
    print(f"S = {S}")
    print(f"σ̂ = {sigma_hat}")
    print(f"Медиана = {median}")
    print(f"Q₁ = {q25}")
    print(f"Q₂ = {q50}")
    print(f"Q₃ = {q75}")
    return x_mean, S2, sigma2_hat, S, sigma_hat, median, q25, q50, q75

In [143]:
sigma_hat = np.std(variation_series_x1, ddof=1)
IQR = np.percentile(variation_series_x1, 75) - np.percentile(variation_series_x1, 25)
xmin, xmax = np.min(variation_series_x1), np.max(variation_series_x1)

# Стёрджес
k_sturges = 1 + int(np.log2(N))

# Скотт
h_scott = 3.5 * sigma_hat * N**(-1/3)
k_scott = int(np.ceil((xmax - xmin) / h_scott))

# Фридман-Диаконис
h_fd = 2 * IQR * N**(-1/3)
k_fd = int(np.ceil((xmax - xmin) / h_fd))

x1_h = h_fd
x1_k = k_fd

print(f'Количество бинов по Стёрджесу: {k_sturges}')
print(f'Количество бинов по Скотту: {k_scott}')
print(f'Количество бинов по Фридману-Диаконису: {k_fd}')

fig = make_subplots(
    rows=3, 
    cols=1,
    subplot_titles=(
        'Стёрджес',
        'Скотт',
        'Фридман-Диаконис'
    ),
    x_title='Значение',
    y_title='Частота'
)

fig.add_trace(
    go.Histogram(
        x=variation_series_x1, 
        xbins=dict(
            start=xmin,
            end=xmax,
            size=(xmax - xmin) / k_sturges 
        ), 
        name='Стёрджес', 
        marker_color='blue'),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=variation_series_x1, 
        xbins=dict(
            start=xmin,
            end=xmax,
            size=h_scott
        ), 
        name='Скотт', 
        marker_color='red'),
    row=2, col=1
)

fig.add_trace(
    go.Histogram(
        x=variation_series_x1, 
        xbins=dict(
            start=xmin,
            end=xmax,
            size=h_fd
        ), 
        name='Фридман-Диаконис', 
        marker_color='orange'),
    row=3, col=1
)

fig.update_layout(
    height=900,
    showlegend=False,
    title_text='Гистограммы с разными методами определения количества бинов'
)

fig.show()

Количество бинов по Стёрджесу: 8
Количество бинов по Скотту: 12
Количество бинов по Фридману-Диаконису: 20


характерна симметрия, отсутсвуют выбросы, отсутствуют естественные границы значений

In [144]:
characteristics_df = pd.DataFrame(columns=['X1', 'X2', 'X3'], index=[
    'Среднее', 'Дисперсия (смещённая)', 'Дисперсия (несмещённая)', 
    'Стандартное отклонение (смещённое)', 'Стандартное отклонение (несмещённое)', 
    'Медиана', 'Q₁', 'Q₂', 'Q₃'
])

print("Значение характеристик для X1:")
x_mean, S2, sigma2_hat, S, sigma_hat, median, q25, q50, q75 = get_characteristics(variation_series_x1)
characteristics_df.loc['Среднее', 'X1'] = x_mean
characteristics_df.loc['Дисперсия (смещённая)', 'X1'] = S2
characteristics_df.loc['Дисперсия (несмещённая)', 'X1'] = sigma2_hat
characteristics_df.loc['Стандартное отклонение (смещённое)', 'X1'] = S
characteristics_df.loc['Стандартное отклонение (несмещённое)', 'X1'] = sigma_hat
characteristics_df.loc['Медиана', 'X1'] = median
characteristics_df.loc['Q₁', 'X1'] = q25
characteristics_df.loc['Q₂', 'X1'] = q50
characteristics_df.loc['Q₃', 'X1'] = q75

Значение характеристик для X1:
x̄ = 7.092266285545157e-05
S² = 0.00014379027671110773
σ̂² = 0.00014451284091568614
S = 0.011991258345607759
σ̂ = 0.012021349379985849
Медиана = -0.00013990291398432764
Q₁ = -0.0060069436002747
Q₂ = -0.00013990291398432764
Q₃ = 0.006371353223412375


In [145]:
sigma_hat = np.std(variation_series_x2, ddof=1)
IQR = np.percentile(variation_series_x2, 75) - np.percentile(variation_series_x2, 25)
xmin, xmax = np.min(variation_series_x2), np.max(variation_series_x2)

# Стёрджес
k_sturges = 1 + int(np.log2(N))

# Скотт
h_scott = 3.5 * sigma_hat * N**(-1/3)
k_scott = int(np.ceil((xmax - xmin) / h_scott))

# Фридман-Диаконис
h_fd = 2 * IQR * N**(-1/3)
k_fd = int(np.ceil((xmax - xmin) / h_fd))

x2_h = h_fd
x2_k = k_fd

print(f'Количество бинов по Стёрджесу: {k_sturges}')
print(f'Количество бинов по Скотту: {k_scott}')
print(f'Количество бинов по Фридману-Диаконису: {k_fd}')

fig = make_subplots(
    rows=3, 
    cols=1,
    subplot_titles=(
        'Стёрджес',
        'Скотт',
        'Фридман-Диаконис'
    ),
    x_title='Значение',
    y_title='Частота'
)

fig.add_trace(
    go.Histogram(
        x=variation_series_x2, 
        xbins=dict(
            start=xmin,
            end=xmax,
            size=(xmax - xmin) / k_sturges 
        ), 
        name='Стёрджес', 
        marker_color='blue'),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=variation_series_x2, 
        xbins=dict(
            start=xmin,
            end=xmax,
            size=h_scott
        ), 
        name='Скотт', 
        marker_color='red'),
    row=2, col=1
)

fig.add_trace(
    go.Histogram(
        x=variation_series_x2, 
        xbins=dict(
            start=xmin,
            end=xmax,
            size=h_fd
        ), 
        name='Фридман-Диаконис', 
        marker_color='orange'),
    row=3, col=1
)

fig.update_layout(
    height=900,
    showlegend=False,
    title_text='Гистограммы с разными методами определения количества бинов'
)

fig.show()

Количество бинов по Стёрджесу: 8
Количество бинов по Скотту: 6
Количество бинов по Фридману-Диаконису: 9


характерна правая асимметрия, присутствуют выбросы в области высоких ставок (90-е годы), есть естественная нижняя граница (ставка ≥ 0)

In [146]:
print("\nЗначение характеристик для X2:")
x_mean, S2, sigma2_hat, S, sigma_hat, median, q25, q50, q75 = get_characteristics(variation_series_x2)
characteristics_df.loc['Среднее', 'X2'] = x_mean
characteristics_df.loc['Дисперсия (смещённая)', 'X2'] = S2
characteristics_df.loc['Дисперсия (несмещённая)', 'X2'] = sigma2_hat
characteristics_df.loc['Стандартное отклонение (смещённое)', 'X2'] = S
characteristics_df.loc['Стандартное отклонение (несмещённое)', 'X2'] = sigma_hat
characteristics_df.loc['Медиана', 'X2'] = median
characteristics_df.loc['Q₁', 'X2'] = q25
characteristics_df.loc['Q₂', 'X2'] = q50
characteristics_df.loc['Q₃', 'X2'] = q75


Значение характеристик для X2:
x̄ = 58.21875
S² = 3673.6243359375
σ̂² = 3692.084759736181
S = 60.61043091694283
σ̂ = 60.76252759502505
Медиана = 25.0
Q₁ = 13.75
Q₂ = 25.0
Q₃ = 80.0


In [147]:
sigma_hat = np.std(variation_series_x3, ddof=1)
IQR = np.percentile(variation_series_x3, 75) - np.percentile(variation_series_x3, 25)
xmin, xmax = np.min(variation_series_x3), np.max(variation_series_x3)

# Стёрджес
k_sturges = 1 + int(np.log2(N))

# Скотт
h_scott = 3.5 * sigma_hat * N**(-1/3)
k_scott = int(np.ceil((xmax - xmin) / h_scott))

# Фридман-Диаконис
h_fd = 2 * IQR * N**(-1/3)
k_fd = int(np.ceil((xmax - xmin) / h_fd))

x3_h = (xmax - xmin) / k_sturges
x3_k = k_sturges

print(f'Количество бинов по Стёрджесу: {k_sturges}')
print(f'Количество бинов по Скотту: {k_scott}')
print(f'Количество бинов по Фридману-Диаконису: {k_fd}')

fig = make_subplots(
    rows=3, 
    cols=1,
    subplot_titles=(
        'Стёрджес',
        'Скотт',
        'Фридман-Диаконис'
    ),
    x_title='Значение',
    y_title='Частота'
)

fig.add_trace(
    go.Histogram(
        x=variation_series_x3, 
        xbins=dict(
            start=xmin,
            end=xmax,
            size=(xmax - xmin) / k_sturges 
        ), 
        name='Стёрджес', 
        marker_color='blue'),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(
        x=variation_series_x3, 
        xbins=dict(
            start=xmin,
            end=xmax,
            size=h_scott
        ), 
        name='Скотт', 
        marker_color='red'),
    row=2, col=1
)

fig.add_trace(
    go.Histogram(
        x=variation_series_x3, 
        xbins=dict(
            start=xmin,
            end=xmax,
            size=h_fd
        ), 
        name='Фридман-Диаконис', 
        marker_color='orange'),
    row=3, col=1
)

fig.update_layout(
    height=900,
    showlegend=False,
    title_text='Гистограммы с разными методами определения количества бинов'
)

fig.show()

Количество бинов по Стёрджесу: 8
Количество бинов по Скотту: 6
Количество бинов по Фридману-Диаконису: 6


характерна симметрия, выбросы отсутствуют, есть естественные границы слева (0) и справа (99)

In [148]:
print("\nЗначение характеристик для X3:")
x_mean, S2, sigma2_hat, S, sigma_hat, median, q25, q50, q75 = get_characteristics(variation_series_x3)
characteristics_df.loc['Среднее', 'X3'] = x_mean
characteristics_df.loc['Дисперсия (смещённая)', 'X3'] = S2
characteristics_df.loc['Дисперсия (несмещённая)', 'X3'] = sigma2_hat
characteristics_df.loc['Стандартное отклонение (смещённое)', 'X3'] = S
characteristics_df.loc['Стандартное отклонение (несмещённое)', 'X3'] = sigma_hat
characteristics_df.loc['Медиана', 'X3'] = median
characteristics_df.loc['Q₁', 'X3'] = q25
characteristics_df.loc['Q₂', 'X3'] = q50
characteristics_df.loc['Q₃', 'X3'] = q75


Значение характеристик для X3:
x̄ = 50.015
S² = 893.6847750000001
σ̂² = 898.1756532663318
S = 29.89456096014792
σ̂ = 29.96957879694561
Медиана = 50.0
Q₁ = 21.5
Q₂ = 50.0
Q₃ = 77.25


В случае с X1 я бы выбрал Гистограмму по Фридману-Диаконису, так как она лучше всего отображает форму распределения данных, детализированно.

В случае с X2 я бы выбрал Гистограмму по Фридману-Диаконису,так как она обеспечивает хорошую детализацию и не слишком шумную картину распределения. 

В случае с X3 я бы выбрал Гистограмму по Стёрджесу,так как она обеспечивает не шумную картину распределения, в отличие от других методов, которые могут создавать слишком много бинов и делать гистограмму слишком детализированной. 

In [149]:
print(f'Ширина бина по Фридману-Диаконису для X1: {x1_h}, количество бинов: {x1_k}')
print(f'Ширина бина по Фридману-Диаконису для X2: {x2_h}, количество бинов: {x2_k}')
print(f'Ширина бина по Стёрджесу для X3: {x3_h}, количество бинов: {x3_k}')

Ширина бина по Фридману-Диаконису для X1: 0.004233317965865892, количество бинов: 20
Ширина бина по Фридману-Диаконису для X2: 22.657181293466238, количество бинов: 9
Ширина бина по Стёрджесу для X3: 12.375, количество бинов: 8


In [150]:
characteristics_df

,X1,X2,X3
Среднее,0.000071,58.21875,50.015
Дисперсия (смещённая),0.000144,3673.624336,893.684775
Дисперсия (несмещённая),0.000145,3692.08476,898.175653
Стандартное отклонение (смещённое),0.011991,60.610431,29.894561
Стандартное отклонение (несмещённое),0.012021,60.762528,29.969579
Медиана,-0.00014,25.0,50.0
Q₁,-0.006007,13.75,21.5
Q₂,-0.00014,25.0,50.0
Q₃,0.006371,80.0,77.25


Пункт 4.2

Гистограмма X1 имеет колоколообразную форму с одной модой вблизи нуля. ЭФР образует характерную S-образную кривую (сигмоид). Распределение симметрично: медиана (−0.00014) практически совпадает со средним (0.000071), квартили расположены симметрично относительно центра (Q₁ = −0.006, Q₃ = 0.006). Данные не имеют естественных границ - лог-доходности теоретически могут принимать любые значения. Всё это соответствует нормальному распределению N(a, σ).

Гистограмма X2 имеет выраженную правую асимметрию: основная масса значений сосредоточена в области 10–80%, но присутствует длинный правый хвост до 210%. ЭФР быстро растёт в начале и замедляется - выпуклая форма, характерная для экспоненциального распределения. Медиана (25.0) значительно меньше среднего (58.2), что типично для правоскошенных распределений. Имеется естественная нижняя граница: минимальная ставка ≈ 10% (сдвиг c). Всё это указывает на экспоненциальное распределение со сдвигом Exp(λ, c).

Гистограмма X3 имеет плоскую форму без выраженного пика - частоты примерно одинаковы по всему диапазону. ЭФР представляет собой почти прямую линию от 0 до 1, что является отличительным признаком равномерного распределения. Среднее (50.0) совпадает с медианой (50.0) и равно середине диапазона [0, 99]. Квартили Q₁ = 21.5 и Q₃ = 77.25 близки к теоретическим 25 и 75. Данные ограничены с обеих сторон: от 0 до 99 (копейки цены). Всё это соответствует равномерному распределению U(a, b).

Пункт 4.3

Методы моментов

In [151]:
mu, sigma = characteristics_df.loc['Среднее', 'X1'], characteristics_df.loc['Стандартное отклонение (несмещённое)', 'X1']
print("По методу моментов приравняли \nEX = μ = x̄ = {},\nDX = σ² = s² = {}.".format(mu, sigma))

По методу моментов приравняли 
EX = μ = x̄ = 7.092266285545157e-05,
DX = σ² = s² = 0.012021349379985849.


По методу моментов приравняли

$EX = μ = x̄ = 0.00007092266285545157$

$DX = σ² = s² = 0.012021349379985849$

In [152]:
# График распределения для X1
x = np.linspace(-0.04, 0.04, 200)
efr = norm.cdf(x, mu, sigma) 


fig_empiric_x1 = go.Figure()
fig_empiric_x1.add_trace(go.Scatter(
    x=variation_series_x1,
    y=empiric_function_x1,
    mode='lines',
    name='Эмпирическая функция распределения', 
    line_shape='hv'
))

fig_empiric_x1.add_trace(go.Scatter(
    x=x, 
    y=efr, 
    mode='lines', 
    name='Функция распределения', 
    line=dict(color='red')
))

fig_empiric_x1.update_xaxes(title_text='x')
fig_empiric_x1.update_yaxes(title_text='Fn(x)')
fig_empiric_x1.update_layout(title='Эмпирическая функция распределения X1')
fig_empiric_x1.show()


In [153]:
# Метод моментов для Exp(λ, c): λ = 1/S, c = x̄ - 1/λ
S_x2 = characteristics_df.loc['Стандартное отклонение (смещённое)', 'X2']
lambda_mm = 1 / S_x2
c_mm = characteristics_df.loc['Среднее', 'X2'] - 1 / lambda_mm

print(f"Метод моментов для Exp(λ, c):")
print(f"λ̂ = 1/S = {lambda_mm:.6f}")
print(f"ĉ = x̄ - 1/λ̂ = {c_mm:.4f}")

# Метод максимального правдоподобия: c = min(x), λ = 1/(x̄ - c)
c_mle = variation_series_x2.min()
lambda_mle = 1 / (characteristics_df.loc['Среднее', 'X2'] - c_mle)

print(f"\nММП для Exp(λ, c):")
print(f"ĉ = min(x) = {c_mle:.4f}")
print(f"λ̂ = 1/(x̄ - ĉ) = {lambda_mle:.6f}")

print(f"\nСравнение:")
print(f"Δλ = {abs(lambda_mm - lambda_mle):.6f}")
print(f"Δc = {abs(c_mm - c_mle):.4f}")


Метод моментов для Exp(λ, c):
λ̂ = 1/S = 0.016499
ĉ = x̄ - 1/λ̂ = -2.3917

ММП для Exp(λ, c):
ĉ = min(x) = 10.0000
λ̂ = 1/(x̄ - ĉ) = 0.020739

Сравнение:
Δλ = 0.004240
Δc = 12.3917


Для $Exp(\lambda, c)$ с плотностью $f(x) = \lambda e^{-\lambda(x-c)}$, $x \geq c$:

$EX = c + \frac{1}{\lambda}, \quad DX = \frac{1}{\lambda^2}$

**Метод моментов:** приравниваем $EX = \bar{x}$, $DX = S^2$:

$$\hat{\lambda}_{MM} = \frac{1}{S}, \quad \hat{c}_{MM} = \bar{x} - \frac{1}{\hat{\lambda}}$$

**ММП:** $\hat{c} = \min(x_i)$, $\hat{\lambda} = \frac{1}{\bar{x} - \hat{c}}$

Оценки различаются: ММ оценивает сдвиг через среднее и дисперсию, а ММП берёт минимум выборки как сдвиг.


In [154]:
# График распределения для X2 (со сдвигом)
from scipy.stats import expon

xx = np.linspace(variation_series_x2.min(), variation_series_x2.max(), 500)

# ФР по методу моментов
y_mm = expon.cdf(xx, loc=c_mm, scale=1/lambda_mm)

# ФР по ММП
y_mle = expon.cdf(xx, loc=c_mle, scale=1/lambda_mle)

empiric_function_x2 = np.arange(1, N+1) / N

fig_empiric_x2 = go.Figure()
fig_empiric_x2.add_trace(go.Scatter(
    x=variation_series_x2,
    y=empiric_function_x2,
    mode='lines',
    line_shape='hv',
    name='ЭФР'
))

fig_empiric_x2.add_trace(go.Scatter(
    x=xx, y=y_mm,
    mode='lines',
    name='ФР (ММ)',
    line=dict(color='red')
))

fig_empiric_x2.add_trace(go.Scatter(
    x=xx, y=y_mle,
    mode='lines',
    name='ФР (ММП)',
    line=dict(color='green')
))

fig_empiric_x2.update_xaxes(title_text='x')
fig_empiric_x2.update_yaxes(title_text='Fn(x)')
fig_empiric_x2.update_layout(title='Эмпирическая функция распределения X2')
fig_empiric_x2.show()


In [155]:
a = characteristics_df.loc['Среднее', 'X3'] - characteristics_df.loc['Стандартное отклонение (несмещённое)', 'X3'] * (3**0.5)
b = characteristics_df.loc['Среднее', 'X3'] + characteristics_df.loc['Стандартное отклонение (несмещённое)', 'X3'] * (3**0.5)

print(f"a = {a}, b = {b}")

a = -1.8938331577487375, b = 101.92383315774873


$EX = \frac{(a + b)}{2}$

$DX = \frac{(b − a)²}{12}$

Из этой системы уравнения получаем, что $a = x̄ − \sqrt{3} · s$, $b = x̄ + \sqrt{3} · s$

$a = -1.8938331577487375$

$b = 101.92383315774873$

далее по формуле $Ф = \frac{x - a}{b - a} = \frac{x + 1.89}{103.82}$


In [156]:
x = np.linspace(0, 99, 200)
y = (x - a) / (b - a)

empiric_function_x3 = np.arange(1, N+1) / N    

fig_empiric_x3 = go.Figure()
fig_empiric_x3.add_trace(go.Scatter(
    x=variation_series_x3,
    y=empiric_function_x3,
    mode='lines',
    line_shape='hv',
    name='Эмпирическая функция распределения'
))

fig_empiric_x3.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='lines',
    name='Функция распределения',
    line=dict(color='red')
))

fig_empiric_x3.update_xaxes(title_text='x')
fig_empiric_x3.update_yaxes(title_text='Fn(x)')
fig_empiric_x3.update_layout(title='Эмпирическая функция распределения X3')
fig_empiric_x3.show()

Метод максимального правдоподобия

### Разберем X1:

Функция правдоподобия для нормального распределения: 
$L(\mu, \sigma^2) = \prod_{i=1}^{n} \frac{1}{\sqrt{2\pi}\sigma} \exp\left(-\frac{(x_i - \mu)^2}{2\sigma^2}\right)$

Логарифм функции правдоподобия:
$\ln{L(\mu, \sigma^2)} = -\frac{n}{2}\ln{2\pi} - \frac{n}{2}\ln{\sigma^2} - \frac{1}{2\sigma^2}\sum_{i=1}^{n}(x_i - \mu)^2$


Производная логарифма по $\mu$: $\frac{\partial \ln{L}}{\partial \mu} = \frac{1}{\sigma^2}\left(\sum_{i=1}^{n}x_i - n\mu\right)$

После приравнивания к 0, получаю: $\hat{\mu} = \frac{\sum_{i=1}^{n}x_i}{n} = \bar{x}$

Производная логарифма по $\sigma^2$: $\frac{\partial \ln{L}}{\partial \sigma^2} = -\frac{n}{2\sigma^2} + \frac{1}{2\sigma^4}\sum_{i=1}^{n}(x_i - \mu)^2$

После приравнивания к 0, получаю: $\hat{\sigma}^2 = \frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2 = S^2$ 

Совпало с методом моментов.

---


### Разберем X2:

Для $Exp(\lambda, c)$ с плотностью $f(x) = \lambda e^{-\lambda(x-c)}$, $x \geq c$:

Функция правдоподобия: 
$L(\lambda, c) = \prod_{i=1}^{n}\lambda e^{-\lambda (x_i - c)} = \lambda^n \exp\left(-\lambda \sum_{i=1}^{n}(x_i - c)\right)$

при условии $c \leq \min(x_i)$

Логарифм: $\ln{L(\lambda, c)} = n\ln{\lambda} - \lambda \sum_{i=1}^{n}(x_i - c)$

По $\lambda$: $\frac{\partial \ln{L}}{\partial \lambda} = \frac{n}{\lambda} - \sum_{i=1}^{n}(x_i - c)$

После приравнивания к 0: $\hat{\lambda} = \frac{n}{\sum(x_i - c)} = \frac{1}{\bar{x} - c}$

По $c$: $\frac{\partial \ln{L}}{\partial c} = n\lambda > 0$ - функция растёт по $c$, значит максимум на границе: $\hat{c} = \min(x_i)$

Итого ММП: $\hat{c} = \min(x_i)$, $\hat{\lambda} = \frac{1}{\bar{x} - \min(x_i)}$

---


In [157]:
variation_series_x3.max(), variation_series_x3.min()

(np.float64(99.0), np.float64(0.0))

### Разберем X3:

Функция правдоподобия для равномерного распределения: 
$L(a, b) = \prod_{i=1}^{n}\frac{1}{b - a} = \frac{1}{(b - a)^n}$

Искать производные нет смысла, так как там не будет 0 => у нас нет максимума внутри, значит он либо на границах, либо нет его.

Чтобы увеличить значени L, нужно уменьшить значение $(b - a)^n$, но у нас есть условие, что $a \leq min(x_i)$ и $b \geq max(x_i)$, но чтобы $(b - a)^n$ было меньше - нужно, чтобы b было меньше, а a - больше. Значит $a = min(x_i) = 0, b = max(x_i) = 99$

тогда функция распределения $Ф = \frac{x - a}{b - a} = \frac{x}{99}$

а мы получали $a = -1.8938331577487375$ и $b = 101.92383315774873$ и $Ф = \frac{x + 1.89}{103.82}$

Сравним тогда их:

---

In [158]:
a = characteristics_df.loc['Среднее', 'X3'] - characteristics_df.loc['Стандартное отклонение (несмещённое)', 'X3'] * (3**0.5)
b = characteristics_df.loc['Среднее', 'X3'] + characteristics_df.loc['Стандартное отклонение (несмещённое)', 'X3'] * (3**0.5)

x = np.linspace(0, 99, 200)
y = (x - a) / (b - a)

empiric_function_x3 = np.arange(1, N+1) / N    

fig_empiric_x3 = go.Figure()
fig_empiric_x3.add_trace(go.Scatter(
    x=variation_series_x3,
    y=empiric_function_x3,
    mode='lines',
    line_shape='hv',
    name='ЭФР'
))

fig_empiric_x3.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='lines',
    name='ФР (ММ)',
    line=dict(color='red')
))

a = 0
b = 99

x = np.linspace(0, 99, 200)
y = (x - a) / (b - a)

fig_empiric_x3.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='lines',
    name='ФР (ММП)',
    line=dict(color='orange')
))

fig_empiric_x3.update_xaxes(title_text='x')
fig_empiric_x3.update_yaxes(title_text='Fn(x)')
fig_empiric_x3.update_layout(title='Эмпирическая функция распределения X3')
fig_empiric_x3.show()

Рассмотрим результаты оценивания параметров.

### X1 (нормальное распределение)

Оценки ММ и ММП совпали:
$$\hat{\mu} = \bar{x}, \quad \hat{\sigma}^2 = S^2 = \frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2$$

Для нормального распределения оба метода приводят к одинаковым формулам.

### X2 (экспоненциальное со сдвигом)

Оценки различаются:

* Метод моментов: $\hat{\lambda}_{MM} = \frac{1}{S}$, $\hat{c}_{MM} = \bar{x} - S$
* ММП: $\hat{c} = \min(x_i)$, $\hat{\lambda} = \frac{1}{\bar{x} - \min(x_i)}$

ММ оценивает сдвиг через среднее и стандартное отклонение, а ММП - через минимум выборки. Оценки λ близки, но оценки c различаются.

### X3 (равномерное распределение)

Оценки различаются:

* Метод моментов: $\hat{a} = \bar{x} - \sqrt{3}S$, $\hat{b} = \bar{x} + \sqrt{3}S$
* ММП: $\hat{a} = \min(x_i)$, $\hat{b} = \max(x_i)$

---

Метод моментов использует числовые характеристики (среднее, дисперсию) и даёт более сглаженные оценки.
ММП учитывает крайние значения выборки и стремится к минимальному интервалу, покрывающему все наблюдения.


Пункт 4.4

Для каждого столбца выбираем порог $x_0 = \bar{x} + \hat{\sigma}$ и оцениваем $P(X > x_0)$ двумя способами.

---

X1 (нормальное)

Эмпирически: считаем долю $\frac{\{x_i > x_0\}}{n}$

Параметрически: $P(X > x_0) = 1 - \Phi\left(\frac{x_0 - \mu}{\sigma}\right)$

где $\Phi$ - функция стандартного нормального распределения, $\mu = \bar{x}$, $\sigma = \hat{\sigma}$


In [159]:
x0_x1 = characteristics_df.loc['Среднее', 'X1'] + characteristics_df.loc['Стандартное отклонение (несмещённое)', 'X1']

p_emp_x1 = np.mean(variation_series_x1 > x0_x1)

mu_x1 = characteristics_df.loc['Среднее', 'X1']
sigma_x1 = characteristics_df.loc['Стандартное отклонение (несмещённое)', 'X1']
p_param_x1 = 1 - norm.cdf(x0_x1, loc=mu_x1, scale=sigma_x1)

print(f"X1 (нормальное)")
print(f"x₀ = x̄ + σ̂ = {mu_x1:.6f} + {sigma_x1:.6f} = {x0_x1:.6f}")
print(f"эмпирическая = {p_emp_x1:.4f}")
print(f"параметрическая = {p_param_x1:.4f}")
print(f"Расхождение = {abs(p_emp_x1 - p_param_x1):.4f}\n")

X1 (нормальное)
x₀ = x̄ + σ̂ = 0.000071 + 0.012021 = 0.012092
эмпирическая = 0.1250
параметрическая = 0.1587
Расхождение = 0.0337



---

X2 (экспоненциальное со сдвигом)

Эмпирически: та же доля

Параметрически: $P(X > x_0) = e^{-\lambda(x_0 - c)}$

где $c = \min(x_i)$, $\lambda = \frac{1}{\bar{x} - c}$ (оценки ММП)

In [160]:
x0_x2 = characteristics_df.loc['Среднее', 'X2'] + characteristics_df.loc['Стандартное отклонение (несмещённое)', 'X2']

p_emp_x2 = np.mean(variation_series_x2 > x0_x2)

c_x2 = variation_series_x2.min()
lambda_x2 = 1 / (characteristics_df.loc['Среднее', 'X2'] - c_x2)
p_param_x2 = np.exp(-lambda_x2 * (x0_x2 - c_x2))

print(f"X2 (экспоненциальное)")
print(f"x₀ = x̄ + σ̂ = {characteristics_df.loc['Среднее', 'X2']:.2f} + {characteristics_df.loc['Стандартное отклонение (несмещённое)', 'X2']:.2f} = {x0_x2:.2f}")
print(f"c = min(x) = {c_x2:.2f}, λ = {lambda_x2:.6f}")
print(f"эмпирическая = {p_emp_x2:.4f}")
print(f"параметрическая = {p_param_x2:.4f}")
print(f"Расхождение = {abs(p_emp_x2 - p_param_x2):.4f}\n")

X2 (экспоненциальное)
x₀ = x̄ + σ̂ = 58.22 + 60.76 = 118.98
c = min(x) = 10.00, λ = 0.020739
эмпирическая = 0.1900
параметрическая = 0.1043
Расхождение = 0.0857



---

X3 (равномерное)

Эмпирически: та же доля

Параметрически: $P(X > x_0) = \frac{b - x_0}{b - a}$

где $a = \min(x_i)$, $b = \max(x_i)$ (оценки ММП)


In [161]:
x0_x3 = characteristics_df.loc['Среднее', 'X3'] + characteristics_df.loc['Стандартное отклонение (несмещённое)', 'X3']

p_emp_x3 = np.mean(variation_series_x3 > x0_x3)

a_x3 = variation_series_x3.min()
b_x3 = variation_series_x3.max()
p_param_x3 = max(0, (b_x3 - x0_x3) / (b_x3 - a_x3))

print(f"=== X3 (равномерное) ===")
print(f"x₀ = x̄ + σ̂ = {characteristics_df.loc['Среднее', 'X3']:.2f} + {characteristics_df.loc['Стандартное отклонение (несмещённое)', 'X3']:.2f} = {x0_x3:.2f}")
print(f"a = min = {a_x3:.2f}, b = max = {b_x3:.2f}")
print(f"эмпирическая = {p_emp_x3:.4f}")
print(f"параметрическая = {p_param_x3:.4f}")
print(f"Расхождение = {abs(p_emp_x3 - p_param_x3):.4f}")

=== X3 (равномерное) ===
x₀ = x̄ + σ̂ = 50.02 + 29.97 = 79.98
a = min = 0.00, b = max = 99.00
эмпирическая = 0.2250
параметрическая = 0.1921
Расхождение = 0.0329


---

Пункт 4.5

Оценка моментов по сгруппированной выборке.

Используем гистограмму (частоты по интервалам и середины интервалов) для оценки $EX$ и $DX$:

$$\hat{X}_g = \frac{1}{n}\sum_{k=1}^{m} n_k \hat{X}_k, \quad \hat{\sigma}^2_g = \frac{1}{n-1}\sum_{k=1}^{m} n_k (\hat{X}_k - \hat{X}_g)^2$$

где $m$ - число интервалов, $n_k$ - число наблюдений в $k$-м интервале, $\hat{X}_k$ - середина $k$-го интервала.

Сравним с оценками по исходным данным.

In [162]:
bins_x1, bins_x2, bins_x3 = x1_k, x2_k, x3_k

for name, x, k in [('X1', variation_series_x1, bins_x1),
                    ('X2', variation_series_x2, bins_x2),
                    ('X3', variation_series_x3, bins_x3)]:
    counts, bin_edges = np.histogram(x, bins=k)
    midpoints = (bin_edges[:-1] + bin_edges[1:]) / 2

    x_g = np.sum(counts * midpoints) / N
    sigma2_g = np.sum(counts * (midpoints - x_g)**2) / (N - 1)

    x_orig = characteristics_df.loc['Среднее', name]
    sigma2_orig = characteristics_df.loc['Дисперсия (несмещённая)', name]

    print(f"{name} (k={k})")
    print(f"По исходным:        EX = {x_orig:.6f},  DX = {sigma2_orig:.6f}")
    print(f"По сгруппированной: EX = {x_g:.6f},  DX = {sigma2_g:.6f}")
    print(f"Отклонение EX: {abs(x_orig - x_g):.6f}")
    print(f"Отклонение DX: {abs(sigma2_orig - sigma2_g):.6f}\n")

X1 (k=20)
По исходным:        EX = 0.000071,  DX = 0.000145
По сгруппированной: EX = 0.000178,  DX = 0.000141
Отклонение EX: 0.000107
Отклонение DX: 0.000004

X2 (k=9)
По исходным:        EX = 58.218750,  DX = 3692.084760
По сгруппированной: EX = 61.222222,  DX = 3388.287115
Отклонение EX: 3.003472
Отклонение DX: 303.797645

X3 (k=8)
По исходным:        EX = 50.015000,  DX = 898.175653
По сгруппированной: EX = 50.551875,  DX = 880.793307
Отклонение EX: 0.536875
Отклонение DX: 17.382346



Оценки по сгруппированной выборке близки к оценкам по исходным данным. Небольшие отклонения связаны с потерей информации при группировке - мы заменяем точные значения серединами интервалов. Чем больше интервалов, тем точнее приближение.

---

Пункт 4.6

Доверительные интервалы ($1 - \alpha = 0.95$)

Асимптотический доверительный интервал для $EX$ (по ЦПТ, для всех столбцов): norm.ppf

Точные доверительные интервалы для нормального распределения (X1):

Для $\mu$ при неизвестной $\sigma^2$: t.ppf
Для $\sigma^2$: есть chi2.ppf

которые потом создают интервал вокруг среднего (мат ожидания)

In [163]:
from scipy.stats import norm, chi2
from scipy.stats import t as t_gen

alpha = 0.05

# Асимптотический ДИ для EX (все столбцы)
t = norm.ppf(1 - alpha / 2)

print(f"Асимптотические ДИ для EX (α = {alpha}):\n")
for name, x in [('X1', variation_series_x1),
                ('X2', variation_series_x2),
                ('X3', variation_series_x3)]:
    x_mean = np.mean(x)
    sigma_hat = np.std(x, ddof=1)
    margin = t * sigma_hat / np.sqrt(N)
    print(f"{name}: ({x_mean - margin:.6f},  {x_mean + margin:.6f}),  ширина = {2*margin:.6f}")

print()

# Точные ДИ для X1 (нормальное)
x = variation_series_x1
x_mean = np.mean(x)
sigma_hat = np.std(x, ddof=1)
sigma2_hat = np.var(x, ddof=1)

# ДИ для μ (t-распределение)
t_crit = t_gen.ppf(1 - alpha / 2, df=N - 1)
margin_t = t_crit * sigma_hat / np.sqrt(N)
print(f"Точный ДИ для μ (X1):  ({x_mean - margin_t:.6f},  {x_mean + margin_t:.6f})")

# ДИ для σ² (χ²-распределение)
chi2_low = chi2.ppf(1 - alpha / 2, df=N - 1)
chi2_high = chi2.ppf(alpha / 2, df=N - 1)
ci_var_low = (N - 1) * sigma2_hat / chi2_low
ci_var_high = (N - 1) * sigma2_hat / chi2_high
print(f"Точный ДИ для σ² (X1): ({ci_var_low:.8f},  {ci_var_high:.8f})")

Асимптотические ДИ для EX (α = 0.05):

X1: (-0.001595,  0.001737),  ширина = 0.003332
X2: (49.797648,  66.639852),  ширина = 16.842204
X3: (45.861505,  54.168495),  ширина = 8.306991

Точный ДИ для μ (X1):  (-0.001605,  0.001747)
Точный ДИ для σ² (X1): (0.00011985,  0.00017771)


Доверительный интервал 95% означает: если бы мы повторили эксперимент множество раз и каждый раз строили ДИ, то в 95% случаев построенный интервал накрыл бы истинное значение параметра.

Это не вероятность того, что параметр лежит в данном конкретном интервале - параметр фиксирован, а интервал случаен.

Асимптотический и точный ДИ для X1 практически совпадают - при $n = 200$ квантили $t$-распределения уже очень близки к квантилям стандартного нормального.

---

Пункт 4.7

Итоговый вывод

X1 (лог-доходности IMOEX) - нормальное распределение $N(\mu, \sigma)$.
Параметры (ММП): $\hat{\mu} = \bar{x}$, $\hat{\sigma} = S$. Гистограмма колоколообразная, ЭФР S-образная, асимметрия 0.

X2 (ставка рефинансирования ЦБ РФ) - экспоненциальное со сдвигом $Exp(\lambda, c)$.
Параметры (ММП): $\hat{c} = \min(x_i)$, $\hat{\lambda} = \frac{1}{\bar{x} - \hat{c}}$. Выраженная правая асимметрия, нижняя граница 10%.

X3 (копейки цены SBER) - равномерное распределение $U(a, b)$.
Параметры (ММП): $\hat{a} = \min(x_i) = 0$, $\hat{b} = \max(x_i) = 99$. Плоская гистограмма, ЭФР - прямая линия.

Доверительные интервалы при $n = 200$ достаточно узкие. Эмпирические и параметрические оценки $P(X > x_0)$ расходятся незначительно, что подтверждает корректность выбранных моделей.

---

Пункт 6. Бонус: бимодальность и кластеризация X4

Построим гистограмму и ЭФР для X4 (цены закрытия IMOEX до и после кризиса 2022).

In [164]:
variation_series_x4 = df['X4'].sort_values(ignore_index=True) if 'X4' in df.columns else pd.Series(np.sort(pd.read_csv('financial_data.csv')['X4'].values))

# Гистограмма X4
fig_hist_x4 = go.Figure()
fig_hist_x4.add_trace(go.Histogram(
    x=variation_series_x4,
    nbinsx=20,
    marker_color='purple'
))
fig_hist_x4.update_layout(title='Гистограмма X4', xaxis=dict(title='Значение'), yaxis=dict(title='Частота'))
fig_hist_x4.show()

# ЭФР X4
empiric_function_x4 = np.arange(1, len(variation_series_x4)+1) / len(variation_series_x4)

fig_efr_x4 = go.Figure()
fig_efr_x4.add_trace(go.Scatter(
    x=variation_series_x4,
    y=empiric_function_x4,
    mode='lines',
    line_shape='hv'
))
fig_efr_x4.update_xaxes(title_text='x')
fig_efr_x4.update_yaxes(title_text='Fn(x)')
fig_efr_x4.update_layout(title='Эмпирическая функция распределения X4')
fig_efr_x4.show()

Гистограмма X4 имеет два выраженных пика - один в районе 2000-2500 (цены после кризиса), второй в районе 3500-4200 (цены до кризиса). ЭФР имеет характерную "ступеньку" посередине - быстрый рост, затем плато, затем снова рост. Это признаки бимодального распределения - смеси двух режимов.

Разделим выборку на два кластера алгоритмом k-средних (k=2).

In [165]:
# K-means для 1D данных (k=2)
def kmeans_1d(x, k=2, max_iter=100):
    centers = np.array([np.percentile(x, 25), np.percentile(x, 75)])
    for _ in range(max_iter):
        dists = np.abs(np.array(x)[:, None] - centers[None, :])
        labels = np.argmin(dists, axis=1)
        new_centers = np.array([np.mean(np.array(x)[labels == i]) for i in range(k)])
        if np.allclose(centers, new_centers):
            break
        centers = new_centers
    return labels, centers

labels, centers = kmeans_1d(variation_series_x4.values)

cluster_0 = variation_series_x4.values[labels == 0]
cluster_1 = variation_series_x4.values[labels == 1]

if np.mean(cluster_0) > np.mean(cluster_1):
    cluster_0, cluster_1 = cluster_1, cluster_0
    centers = centers[::-1]

print(f"Кластер 0 (после кризиса): n={len(cluster_0)}, mean={np.mean(cluster_0):.1f}, std={np.std(cluster_0, ddof=1):.1f}")
print(f"Кластер 1 (до кризиса):    n={len(cluster_1)}, mean={np.mean(cluster_1):.1f}, std={np.std(cluster_1, ddof=1):.1f}")
print(f"Центры: {centers[0]:.1f}, {centers[1]:.1f}")
print(f"\nОбщее среднее X4: {np.mean(variation_series_x4):.1f}")

Кластер 0 (после кризиса): n=100, mean=2300.2, std=152.7
Кластер 1 (до кризиса):    n=100, mean=3920.8, std=145.9
Центры: 2300.2, 3920.8

Общее среднее X4: 3110.5


In [166]:
fig_clusters = go.Figure()

fig_clusters.add_trace(go.Histogram(
    x=cluster_0,
    nbinsx=15,
    name=f'Кластер 0 (n={len(cluster_0)})',
    marker_color='steelblue',
    opacity=0.7
))

fig_clusters.add_trace(go.Histogram(
    x=cluster_1,
    nbinsx=15,
    name=f'Кластер 1 (n={len(cluster_1)})',
    marker_color='coral',
    opacity=0.7
))

fig_clusters.add_vline(x=centers[0], line_dash="dash", line_color="blue",
                       annotation_text=f"Центр 0: {centers[0]:.0f}")
fig_clusters.add_vline(x=centers[1], line_dash="dash", line_color="red",
                       annotation_text=f"Центр 1: {centers[1]:.0f}")
fig_clusters.add_vline(x=np.mean(variation_series_x4), line_dash="dot", line_color="gray",
                       annotation_text=f"Общее среднее: {np.mean(variation_series_x4):.0f}")

fig_clusters.update_layout(
    barmode='overlay',
    title='X4: кластеры',
    xaxis_title='Значение',
    yaxis_title='Частота'
)
fig_clusters.show()

In [167]:
fig_box = go.Figure()
fig_box.add_trace(go.Box(y=cluster_0, name='Кластер 0 (после кризиса)', marker_color='steelblue'))
fig_box.add_trace(go.Box(y=cluster_1, name='Кластер 1 (до кризиса)', marker_color='coral'))
fig_box.update_layout(title='X4 Усы по кластерам', yaxis_title='Значение')
fig_box.show()

---

Общее среднее X4 плохо описывает данные, потому что оно оказывается между двумя модами и не соответствует ни одному из реальных режимов рынка.

Кластер 0 - цены IMOEX после кризиса 2022 (≈2000–2800), кластер 1 - до кризиса (≈3700–4300). Это два принципиально разных состояния рынка.

Если описывать каждый кластер отдельно нормальным распределением, подгонка будет значительно лучше, чем любая одномодальная модель для всей выборки.